
## 01MIAR - Actividad Video Valencia Pollution - Code

*Ivan Fuertes*

## Web API pública
- Interfaz de programación que se pone a disposición de cualquier desarrollador para acceder a datos o funcionalidades
### Valencia Open Data
- https://opendata.vlci.valencia.es/
#### Estaciones contaminación atmosférica
- https://opendata.vlci.valencia.es/dataset/estacions-contaminacio-atmosferiques-estaciones-contaminacion-atmosfericas

#### API Creada con CKAN
- https://docs.ckan.org/en/2.11/

### Requests
- Librería para hacer peticiones Http a web
- https://requests.readthedocs.io/en/latest/

In [2]:
import requests

In [3]:
# function to get a json response from a url + params

def make_request(base_url, params):
    response = requests.get(base_url, params = params)
    if response:
        return response.json()
    else:
        print(response)
        raise Exception("Error Downloading JSON")

- URL Base del servidor
- https://geoportal.valencia.es/server/rest/services/OPENDATA/

In [4]:
server_url = "https://geoportal.valencia.es/server/rest/services/OPENDATA/"

- Prueba de la llamada a la API
- https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query?where=1=1&outFields=*&f=json

In [5]:
base_url = server_url + "MedioAmbiente/MapServer/156/query"

#### Parámetros del Query
- `'f' = 'json'` - formato del fichero de respuesta
- `'outFields' = '*'` - campos en la respuesta, todos
- `'where' = '1=1'` - filtro

In [6]:
params = {'f' : 'json', 'outFields' : '*', 'where' : '1=1'}
print(base_url)

https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query


#### Realizar petición y explorar resultados
- `'fieldAliases'` = nombres de campos y sus alias
- `'fields'` = nombres de campos, alias, tipos y longitudes
- `'features'` = lista de estaciones y lecturas
  - `'attributes'` = lectura de una estación
  - `'geometry'` = coordenadas geométricas de una estación (sistema proyectado UTM)

In [7]:
import json   # to acquire and format json string

response = make_request(base_url, params)

print(json.dumps(response, indent = 2))

{
  "displayFieldName": "nombre",
  "fieldAliases": {
    "objectid": "objectid",
    "nombre": "Nom / Nombre",
    "direccion": "Adre\u00e7a / Direccion",
    "tipozona": "Tipus Zona / Tipo Zona",
    "parametros": "Par\u00e0metres / Par\u00e1metros",
    "mediciones": "Mesuraments / Mediciones",
    "so2": "so2",
    "no2": "no2",
    "o3": "o3",
    "co": "co",
    "pm10": "pm10",
    "pm25": "pm25",
    "tipoemisio": "tipoemision",
    "fecha_carg": "fecha_carga",
    "calidad_am": "calidad_ambiental",
    "fiwareid": "fiwareid"
  },
  "geometryType": "esriGeometryPoint",
  "spatialReference": {
    "wkid": 25830,
    "latestWkid": 25830
  },
  "fields": [
    {
      "name": "objectid",
      "type": "esriFieldTypeOID",
      "alias": "objectid"
    },
    {
      "name": "nombre",
      "type": "esriFieldTypeString",
      "alias": "Nom / Nombre",
      "length": 254
    },
    {
      "name": "direccion",
      "type": "esriFieldTypeString",
      "alias": "Adre\u00e7a / Direcci

In [8]:
from pyproj import Transformer   # to converto from UTM to GPS lat+long
import pandas as pd

transformer = Transformer.from_crs("EPSG:32630", "EPSG:4326", always_xy=True)

df = pd.DataFrame()

for feature in response['features']:
    serie = pd.Series(feature['attributes'])

    # transform from UTM to GPS
    serie["lon"], serie["lat"] = transformer.transform(feature['geometry']['x'], feature['geometry']['y'])
    
    df = pd.concat([df, serie.to_frame().T], axis = 0, ignore_index = True)

# format datetime from epoch (as miliseconds) to timestamp (GMT)
df['fecha_carg'] = pd.to_datetime(df['fecha_carg'], unit = 'ms')

In [9]:
display(df.head(3))

,objectid,nombre,direccion,tipozona,parametros,mediciones,so2,no2,o3,co,pm10,pm25,tipoemisio,fecha_carg,calidad_am,fiwareid,lon,lat
0,18,Viveros,VIVERS,Urbana,"Dióxido de azufre (SO2),Óxidos de nitrógeno to...",http://mapas.valencia.es/WebsMunicipales/uploa...,3.0,3.0,127.0,None,None,None,Fondo,2026-05-30 14:00:00,Regular,A06_VIVERS_60m,-0.369648,39.479641
1,19,Centro,VALÈNCIA CENTRE,Urbana,"Óxidos de nitrógeno totales (NOx),Monóxido de ...",https://mapas.valencia.es/WebsMunicipales/uplo...,None,6.0,None,None,29.0,12.0,Tráfico,2026-05-30 14:00:00,Razonablemente Buena,A07_VALENCIACENTRE_60m,-0.376398,39.470548
2,22,Patraix,PATRAIX,Urbana,"Óxidos de nitrógeno totales (NOx),Monóxido de ...",None,None,53.0,None,None,29.0,13.0,Tráfico,2026-05-30 14:00:00,Razonablemente Buena,A11_PATRAIX_60m,-0.401411,39.459189


### Tomar muestras de los datos cada hora durante un día completo

In [10]:
sleep_time = 60 * 60 # 60 minutes in seconds
total_time = (24 * 60 * 60) + sleep_time  # 1 day to take samples (plus one sleep extra just in case)
current_time = 0 # current time before ending sampling

In [11]:
# endpoints
server_url = "https://geoportal.valencia.es/server/rest/services/OPENDATA/"
base_url = server_url + "MedioAmbiente/MapServer/156/query"

In [12]:
# params
params = {'f' : 'json', 'outFields' : '*', 'where' : '1=1'}

In [13]:
# path to save dataset in csv format
import os
from time import sleep

path_csv = ['res', 'valencia_pollution_dataset.csv']
path_csv_solved = os.path.join(*path_csv)
first_time = True # flag for one use only to save headers in csv

In [14]:
# main loop
current_time=0
while current_time < total_time:
    pollution_list = make_request(base_url, params)['features']

    print(f"Current Time = {current_time}, Records Processed = {len(pollution_list)}")
    
    df = pd.DataFrame()
    for feature in pollution_list:
        serie = pd.Series(feature['attributes'])
        serie["lon"], serie["lat"] = transformer.transform(feature['geometry']['x'], feature['geometry']['y'])
        df = pd.concat([df, serie.to_frame().T], axis = 0, ignore_index = True)
        
    df['fecha_carg'] = pd.to_datetime(df['fecha_carg'], unit = 'ms')

    df.to_csv(path_csv_solved, sep=',', header=first_time, mode='a', index=False)

    first_time = False
    sleep(sleep_time)
    current_time += sleep_time

Current Time = 0, Records Processed = 11


OSError: Cannot save file into a non-existent directory: 'res'

In [15]:
display(df.head(5))

,objectid,nombre,direccion,tipozona,parametros,mediciones,so2,no2,o3,co,pm10,pm25,tipoemisio,fecha_carg,calidad_am,fiwareid,lon,lat
0,20,Cabanyal,CABANYAL,Urbana,None,None,None,14.0,None,None,34.0,12.0,Fondo,2026-05-29 21:00:00,Razonablemente Buena,A09_CABANYAL_60m,-0.328535,39.474391
1,21,Olivereta,OLIVERETA,Urbana,"Óxidos de nitrógeno totales (NOx),Monóxido de ...",None,None,37.0,None,None,50.0,17.0,Tráfico,2026-05-29 21:00:00,Regular,A10_OLIVERETA_60m,-0.405923,39.469244
2,14,Boulevar Sur,BULEVARD SUD,Urbana,"Dióxido de azufre (SO2),Ozono,Óxidos de nitróg...",http://mapas.valencia.es/WebsMunicipales/uploa...,1.0,22.0,94.0,None,None,None,Tráfico,2026-05-29 21:00:00,Razonablemente Buena,A02_BULEVARDSUD_60m,-0.396338,39.450396
3,15,Molí del Sol,MOLÍ DEL SOL,Suburbana,"Dióxido de azufre (SO2),Monóxido de carbono (C...",http://mapas.valencia.es/WebsMunicipales/uploa...,3.0,12.0,98.0,0.0,38.0,10.0,Tráfico,2026-05-29 21:00:00,Razonablemente Buena,A03_MOLISOL_60m,-0.40881,39.481112
4,17,Universidad Politécnica,POLITÈCNIC,Suburbana,"Dióxido de azufre (SO2),Ozono,Óxidos de nitróg...",http://mapas.valencia.es/WebsMunicipales/uploa...,1.0,13.0,90.0,None,19.0,9.0,Fondo,2026-05-29 21:00:00,Razonablemente Buena,A05_POLITECNIC_60m,-0.337401,39.479644


In [20]:
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   objectid    11 non-null     object        
 1   nombre      11 non-null     object        
 2   direccion   11 non-null     object        
 3   tipozona    11 non-null     object        
 4   parametros  10 non-null     object        
 5   mediciones  7 non-null      object        
 6   so2         6 non-null      object        
 7   no2         11 non-null     object        
 8   o3          6 non-null      object        
 9   co          3 non-null      object        
 10  pm10        9 non-null      object        
 11  pm25        9 non-null      object        
 12  tipoemisio  11 non-null     object        
 13  fecha_carg  11 non-null     datetime64[ms]
 14  calidad_am  11 non-null     object        
 15  fiwareid    11 non-null     object        
 16  lon         11 non-null     object     

None

In [21]:
display(df.describe(include='all'))

,objectid,nombre,direccion,tipozona,parametros,mediciones,so2,no2,o3,co,pm10,pm25,tipoemisio,fecha_carg,calidad_am,fiwareid,lon,lat
count,11.0,11,11,11,10,7,6.0,11.0,6.0,3.0,9.0,9.0,11,11,11,11,11.000000,11.000000
unique,11.0,11,11,2,8,7,3.0,8.0,6.0,1.0,7.0,8.0,2,NaN,2,11,11.000000,11.000000
top,20.0,Cabanyal,CABANYAL,Urbana,"Óxidos de nitrógeno totales (NOx),Monóxido de ...",http://mapas.valencia.es/WebsMunicipales/uploa...,3.0,14.0,94.0,0.0,38.0,12.0,Tráfico,NaN,Razonablemente Buena,A09_CABANYAL_60m,-0.328535,39.474391
freq,1.0,1,1,9,3,1,3.0,3.0,1.0,3.0,3.0,2.0,8,NaN,8,1,1.000000,1.000000
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-29 21:00:00,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-29 21:00:00,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-29 21:00:00,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-29 21:00:00,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-29 21:00:00,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-29 21:00:00,NaN,NaN,NaN,NaN
